# Project flow

This notebook shows the calculation order and final portfolio totals.

In [1]:
from pathlib import Path
import json
import sqlite3
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score, brier_score_loss, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().resolve()
if not (ROOT / "config").exists():
    ROOT = ROOT.parent
DB = ROOT / "database" / "ifrs9_ecl.sqlite3"

def query(sql):
    with sqlite3.connect(DB) as connection:
        return pd.read_sql_query(sql, connection)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
plt.rcParams["figure.figsize"] = (9, 4)

Calculation order

1. Load loan and macro history.
2. Define default and build model features.
3. Estimate 12-month PD and monthly hazard.
4. Assign stages and build macro scenarios.
5. Create monthly PD, LGD, EAD and discount-factor terms.
6. Calculate scenario ECL and apply scenario weights.
7. Reconcile loan, stage and portfolio totals.

In [2]:
query("select * from executive_summary")

,metric,value,unit
0,Reporting date,2026-03-01,date
1,Mortgage accounts,3471,count
2,Gross exposure,432963771.97,USD
3,Probability-weighted ECL,424001.865485687,USD
4,Portfolio coverage ratio,0.00097930102455562,percent
5,Exposure-weighted 12-month PD,0.0118070162920678,percent
6,Exposure-weighted average LGD,0.0232144463924122,percent


In [3]:
query("select * from stage_summary order by stage")

,stage,loans,gross_exposure,ecl,coverage_ratio,exposure_share,ecl_share
0,1,192,"28,490,378.8900","1,105.3745",0.0000,0.0658,0.0026
1,2,3261,"401,710,712.7500","387,650.6954",0.0010,0.9278,0.9143
2,3,18,"2,762,680.3300","35,245.7956",0.0128,0.0064,0.0831


In [4]:
query("select * from scenario_summary order by scenario")

,scenario,scenario_weight,scenario_ecl,weighted_contribution
0,Base,0.5798,"419,658.5693","243,320.2848"
1,Downside,0.1693,"458,859.3606","77,704.6508"
2,Upside,0.2509,"410,509.3905","102,976.9298"


The detailed calculation is stored in ecl_projection_cube. Each row is one loan, one scenario and one future month.